# EDA Project: Exploratory Analysis of House Prices

**Dataset:** Ames Housing, Kaggle "House Prices: Advanced Regression Techniques" (`train.csv`), 1,460 rows × 81 columns
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy
**Objective:** Use exploratory data analysis only (no machine learning) to understand which factors drive house sale prices.

## Guiding questions
1. How is `SalePrice` distributed? Is it skewed?
2. Which numerical features correlate most strongly with `SalePrice`?
3. Does `OverallQual` change price in a clear, ordered way?
4. Do some neighborhoods consistently cost more than others?
5. Are there outliers, and are they errors or real but unusual sales?
6. Which columns have missing values, and which of those are not truly missing?

## Notebook structure
1. Introduction  2. Data loading  3. Data understanding  4. Data cleaning  5. Univariate analysis  6. Bivariate and multivariate analysis  7. Outlier analysis  8. Dashboard  9. Conclusions

## 2. Data loading

**Step 1: Setup.** Import the libraries and set a consistent plotting style and colour palette.

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

BLUE, ORANGE = "#2a6f97", "#e07a1f"                       # consistent palette
price_fmt = FuncFormatter(lambda x, _: f"${x/1000:.0f}k")  # axis labels like $200k

**Step 1: First look.** Load `train.csv`, check shape, first rows, data types, and which columns have missing values.

In [3]:
import os
if not os.path.exists("train.csv"):
    from google.colab import files
    files.upload()          # a file picker opens: choose train.csv

df = pd.read_csv("train.csv")
print("Shape:", df.shape)
display(df.head())
df.info()

missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing_table = pd.DataFrame({"missing_count": missing,
                              "missing_pct": (missing / len(df) * 100).round(2)})
print(f"\n{len(missing)} columns have missing values:")
display(missing_table)

Shape: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.00,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.00,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.00,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.00,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.00,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.00,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.00,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.00,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.00,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.00,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.00,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.00,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.00,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.00,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.00,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

,missing_count,missing_pct
PoolQC,1453,99.52
MiscFeature,1406,96.30
Alley,1369,93.77
Fence,1179,80.75
MasVnrType,872,59.73
FireplaceQu,690,47.26
LotFrontage,259,17.74
GarageQual,81,5.55
GarageFinish,81,5.55
GarageType,81,5.55


### What I learned
- The data has **1,460 rows and 81 columns**: 35 integer, 3 float and 43 text columns.
- **19 columns** contain missing values. The biggest are `PoolQC` (99.5%), `MiscFeature` (96.3%), `Alley` (93.8%), `Fence` (80.8%), `MasVnrType` (59.7%), `FireplaceQu` (47.3%) and `LotFrontage` (17.7%).
- First impression: a house with no pool or no alley is not "missing data", so many of these gaps probably mean "feature absent". I will test this in the cleaning section.

## 3. Data understanding

**Step 2: Classify variables.** Split columns into numerical (continuous or discrete), nominal, and ordinal. Convert `ExterQual` and `KitchenQual` to ordered categories (Po < Fa < TA < Gd < Ex) and build a summary table.

In [4]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(exclude="number").columns.tolist()
print(len(numeric_cols), "numeric columns |", len(categorical_cols), "text columns")

quality_order = ["Po", "Fa", "TA", "Gd", "Ex"]
ordinal_map = {"ExterQual": quality_order, "KitchenQual": quality_order}
for col, order in ordinal_map.items():
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)

discrete_cols = ["YearBuilt", "YearRemodAdd", "GarageYrBlt", "YrSold", "MoSold",
                 "FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath", "BedroomAbvGr",
                 "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageCars"]
ordinal_cols = ["OverallQual", "OverallCond"] + list(ordinal_map)

def var_type(col):
    if col == "Id": return "Identifier"
    if col in ordinal_cols: return "Ordinal"
    if col == "MSSubClass" or col in categorical_cols: return "Categorical (nominal)"
    if col in discrete_cols: return "Numerical (discrete)"
    return "Numerical (continuous)"

summary = pd.DataFrame({
    "column": df.columns,
    "type": [var_type(c) for c in df.columns],
    "n_unique": [df[c].nunique() for c in df.columns],
    "examples": [", ".join(map(str, df[c].dropna().unique()[:3])) for c in df.columns]})

print(summary["type"].value_counts())
display(summary)

38 numeric columns | 43 text columns
type
Categorical (nominal)     42
Numerical (continuous)    20
Numerical (discrete)      14
Ordinal                    4
Identifier                 1
Name: count, dtype: int64


,column,type,n_unique,examples
0,Id,Identifier,1460,"1, 2, 3"
1,MSSubClass,Categorical (nominal),15,"60, 20, 70"
2,MSZoning,Categorical (nominal),5,"RL, RM, C (all)"
3,LotFrontage,Numerical (continuous),110,"65.0, 80.0, 68.0"
4,LotArea,Numerical (continuous),1073,"8450, 9600, 11250"
5,Street,Categorical (nominal),2,"Pave, Grvl"
6,Alley,Categorical (nominal),2,"Grvl, Pave"
7,LotShape,Categorical (nominal),4,"Reg, IR1, IR2"
8,LandContour,Categorical (nominal),4,"Lvl, Bnk, Low"
9,Utilities,Categorical (nominal),2,"AllPub, NoSeWa"


### What I learned
- Of the 81 columns: **42 nominal, 20 continuous, 14 discrete, 4 ordinal, 1 identifier** (`Id`).
- `OverallQual` and `OverallCond` are stored as numbers but are really **ordinal** (a quality of 8 is better than 4, but not "twice as good").
- `MSSubClass` is also stored as a number, but it is a **nominal code** for building type, so averaging it makes no sense.
- `ExterQual` and `KitchenQual` have no "Po" values in this data, but I kept "Po" in the category order so the scale is complete.

**Step 3: Descriptive statistics.** Calculate mean, median, mode, spread, skewness and kurtosis for `SalePrice`, `GrLivArea` and `LotArea`, and count the categories in `Neighborhood` and `HouseStyle`.

In [5]:
display(df.describe().T)

rows = []
for col in ["SalePrice", "GrLivArea", "LotArea"]:
    s = df[col]
    rows.append({"column": col, "mean": s.mean(), "median": s.median(),
                 "mode": s.mode().iloc[0], "std": s.std(), "variance": s.var(),
                 "range": s.max() - s.min(), "IQR": s.quantile(0.75) - s.quantile(0.25),
                 "skewness": s.skew(), "kurtosis": s.kurt()})
stats_table = pd.DataFrame(rows).set_index("column")
display(stats_table.T)

for col in stats_table.index:
    m, md = stats_table.loc[col, "mean"], stats_table.loc[col, "median"]
    note = "right skewed" if m > md else "left skewed" if m < md else "symmetric"
    print(f"{col}: mean {m:,.0f} vs median {md:,.0f} -> {note}")

for col in ["Neighborhood", "HouseStyle"]:
    print(f"\n{col} - mode: {df[col].mode().iloc[0]}")
    print(df[col].value_counts())

,count,mean,std,min,25%,50%,75%,max
Id,1460.00,730.50,421.61,1.00,365.75,730.50,1095.25,1460.00
MSSubClass,1460.00,56.90,42.30,20.00,20.00,50.00,70.00,190.00
LotFrontage,1201.00,70.05,24.28,21.00,59.00,69.00,80.00,313.00
LotArea,1460.00,10516.83,9981.26,1300.00,7553.50,9478.50,11601.50,215245.00
OverallQual,1460.00,6.10,1.38,1.00,5.00,6.00,7.00,10.00
OverallCond,1460.00,5.58,1.11,1.00,5.00,5.00,6.00,9.00
YearBuilt,1460.00,1971.27,30.20,1872.00,1954.00,1973.00,2000.00,2010.00
YearRemodAdd,1460.00,1984.87,20.65,1950.00,1967.00,1994.00,2004.00,2010.00
MasVnrArea,1452.00,103.69,181.07,0.00,0.00,0.00,166.00,1600.00
BsmtFinSF1,1460.00,443.64,456.10,0.00,0.00,383.50,712.25,5644.00


column,SalePrice,GrLivArea,LotArea
mean,180921.20,1515.46,10516.83
median,163000.00,1464.00,9478.50
mode,140000.00,864.00,7200.00
std,79442.50,525.48,9981.26
variance,6311111264.30,276129.63,99625649.65
range,720100.00,5308.00,213945.00
IQR,84025.00,647.25,4048.00
skewness,1.88,1.37,12.21
kurtosis,6.54,4.90,203.24


SalePrice: mean 180,921 vs median 163,000 -> right skewed
GrLivArea: mean 1,515 vs median 1,464 -> right skewed
LotArea: mean 10,517 vs median 9,478 -> right skewed

Neighborhood - mode: NAmes
Neighborhood
NAmes      225
CollgCr    150
OldTown    113
Edwards    100
Somerst     86
Gilbert     79
NridgHt     77
Sawyer      74
NWAmes      73
SawyerW     59
BrkSide     58
Crawfor     51
Mitchel     49
NoRidge     41
Timber      38
IDOTRR      37
ClearCr     28
SWISU       25
StoneBr     25
Blmngtn     17
MeadowV     17
BrDale      16
Veenker     11
NPkVill      9
Blueste      2
Name: count, dtype: int64

HouseStyle - mode: 1Story
HouseStyle
1Story    726
2Story    445
1.5Fin    154
SLvl       65
SFoyer     37
1.5Unf     14
2.5Unf     11
2.5Fin      8
Name: count, dtype: int64


### What I learned
- `SalePrice`: mean **$180,921** vs median **$163,000**. The mean is about $18,000 (11%) higher, so a few very expensive houses pull the mean up. This is a **right-skewed** distribution (skewness 1.88, kurtosis 6.54).
- `GrLivArea` is mildly right skewed (skewness 1.37). `LotArea` is extremely skewed (skewness 12.2, kurtosis 203): the largest lot is 215,245 sq ft against a median of 9,478.
- The most common neighborhood is **NAmes** (225 houses). The most common house style is **1Story** (726 of 1,460, about half).
- Interpretation: when mean > median, the long tail is on the high side. For prices, that means a small number of luxury homes.

## 4. Data cleaning

**Step 9 (run early, because it affects later steps): duplicates and inconsistent labels.** Check for duplicate rows, then check text columns for casing problems, extra spaces and similar spellings.

In [6]:
dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)
if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("New shape:", df.shape)

problems = []
for col in categorical_cols:
    labels = df[col].dropna().astype(str).unique()
    cleaned = pd.Series(labels).str.strip().str.lower().unique()
    if len(cleaned) != len(labels):
        problems.append(col)
print("Columns with casing/extra-space problems:", problems if problems else "none")

# Similar spellings: Exterior1st and Exterior2nd describe the same materials
print("\nIn Exterior2nd but not Exterior1st:", sorted(set(df["Exterior2nd"]) - set(df["Exterior1st"])))
print("In Exterior1st but not Exterior2nd:", sorted(set(df["Exterior1st"]) - set(df["Exterior2nd"])))

exterior_fix = {"Brk Cmn": "BrkComm", "CmentBd": "CemntBd", "Wd Shng": "WdShing"}
df["Exterior2nd"] = df["Exterior2nd"].replace(exterior_fix)
print("\nAfter fix, only in Exterior2nd:", sorted(set(df["Exterior2nd"]) - set(df["Exterior1st"])))

Duplicate rows: 0
Columns with casing/extra-space problems: none

In Exterior2nd but not Exterior1st: ['Brk Cmn', 'CmentBd', 'Other', 'Wd Shng']
In Exterior1st but not Exterior2nd: ['BrkComm', 'CemntBd', 'WdShing']

After fix, only in Exterior2nd: ['Other']


### What I learned
- **No duplicate rows** and **no casing or extra-space problems**.
- There is a real spelling inconsistency: `Exterior2nd` writes three materials differently from `Exterior1st` (`Brk Cmn` vs `BrkComm`, `CmentBd` vs `CemntBd`, `Wd Shng` vs `WdShing`). I standardised them with a mapping dictionary.
- `Other` now appears only in `Exterior2nd`, which is a valid category, so I left it.

**Step 9: Which "missing" values are really missing?** First I re-read the raw file without pandas' automatic NA conversion to see what the file actually contains. Then I cross-check each column against a related column (for example, `PoolQC` is blank, so is `PoolArea` equal to 0?).

In [ ]:
raw = pd.read_csv("train.csv", keep_default_na=False)
proof = pd.DataFrame({
    "NaN_in_df": df[missing_table.index].isnull().sum(),
    'literal "NA" in file': [(raw[c] == "NA").sum() for c in missing_table.index],
    'literal "None" in file': [(raw[c] == "None").sum() for c in missing_table.index]})
display(proof)

absent_rules = {
    "PoolQC": df["PoolArea"] == 0,
    "FireplaceQu": df["Fireplaces"] == 0,
    "GarageType": df["GarageArea"] == 0, "GarageFinish": df["GarageArea"] == 0,
    "GarageQual": df["GarageArea"] == 0, "GarageCond": df["GarageArea"] == 0,
    "BsmtQual": df["TotalBsmtSF"] == 0, "BsmtCond": df["TotalBsmtSF"] == 0,
    "BsmtExposure": df["TotalBsmtSF"] == 0, "BsmtFinType1": df["TotalBsmtSF"] == 0,
    "BsmtFinType2": df["TotalBsmtSF"] == 0}
check = pd.DataFrame({
    "NaN rows": {c: df[c].isnull().sum() for c in absent_rules},
    "NaN AND feature absent": {c: (df[c].isnull() & m).sum() for c, m in absent_rules.items()}})
check["truly unknown"] = check["NaN rows"] - check["NaN AND feature absent"]
display(check)

### What I learned
- In the raw file these blanks are the text **"NA"**, which the data description defines as "no such feature", but pandas reads it as NaN.
- The cross-check confirms it: all **1,453** `PoolQC` blanks have `PoolArea = 0`, all **690** `FireplaceQu` blanks have 0 fireplaces, all **81** garage blanks have `GarageArea = 0`, and **37** basement blanks have `TotalBsmtSF = 0`.
- Only **1** `BsmtExposure` and **1** `BsmtFinType2` value are truly unknown.
- `MasVnrType` is a different case. Most of its blanks are the literal word **"None"** (no masonry veneer) that pandas also reads as missing. Only 8 are real "NA". If your pandas version already shows fewer blanks there, that is the same effect.
- `LotFrontage` (259) and `Electrical` (1) are **genuinely missing** values.
- This answers guiding question 6: dropping these columns would throw away real information.

**Step 9: Fill missing values and document every decision.**

| Columns | Decision | Reason |
|---|---|---|
| PoolQC, MiscFeature, Alley, Fence, FireplaceQu, Garage*, Bsmt*, MasVnrType | fill with `"None"` | NA means the feature is absent |
| Electrical (1 row) | fill with the mode | genuinely missing, only one row |
| MasVnrArea | fill with 0 | no veneer means zero area |
| GarageYrBlt | fill with `YearBuilt` and add a `HasGarage` flag | no garage means no year; the flag keeps that information |
| LotFrontage | median of the same neighborhood | frontage differs a lot between neighborhoods |

I also check for impossible values (build year after sale year, negative areas).

In [ ]:
none_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
             "GarageType", "GarageFinish", "GarageQual", "GarageCond",
             "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
             "MasVnrType"]
for col in none_cols:
    df[col] = df[col].fillna("None")

df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode().iloc[0])
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["YearBuilt"])
df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(lambda s: s.fillna(s.median()))
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

print("Remaining missing values:", df.isnull().sum().sum())

area_cols = [c for c in numeric_cols if "SF" in c or "Area" in c]
impossible = {
    "YearBuilt > YrSold": (df["YearBuilt"] > df["YrSold"]).sum(),
    "YearRemodAdd < YearBuilt": (df["YearRemodAdd"] < df["YearBuilt"]).sum(),
    "GarageYrBlt > YrSold": (df["GarageYrBlt"] > df["YrSold"]).sum(),
    "Negative area / SF values": (df[area_cols] < 0).sum().sum()}
print(pd.Series(impossible))

### What I learned
- Remaining missing values: **0**. No impossible values were found (no house sold before it was built, no negative areas).
- The 2 truly unknown basement values and the 8 unknown `MasVnrType` values are filled with "None". That is a small assumption affecting under 1% of rows, and I note it here as a limitation.
- I added one new column, `HasGarage`, so `df` now has 82 columns.

## 5. Univariate analysis

**Step 4: Distribution of the target.** Plot `SalePrice` as a histogram with KDE, as a boxplot, and after a log transform, then compare skewness.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df["SalePrice"], kde=True, bins=40, color=BLUE, ax=axes[0])
axes[0].set(title="Distribution of SalePrice", xlabel="Sale price ($)", ylabel="Number of houses")

sns.boxplot(y=df["SalePrice"], color=BLUE, ax=axes[1])
axes[1].set(title="Boxplot of SalePrice", ylabel="Sale price ($)")

sns.histplot(np.log(df["SalePrice"]), kde=True, bins=40, color=ORANGE, ax=axes[2])
axes[2].set(title="Distribution of log(SalePrice)", xlabel="log(Sale price)", ylabel="Number of houses")

plt.tight_layout()
plt.show()

print(f"Skewness before log: {df['SalePrice'].skew():.3f}")
print(f"Skewness after log : {np.log(df['SalePrice']).skew():.3f}")

### What I learned
- `SalePrice` is clearly right skewed: most houses cluster between about $100k and $250k with a long tail towards $755k.
- The boxplot shows many points above the upper whisker, all on the expensive side.
- A log transform reduces skewness from **1.88 to 0.12**, so the log of price is close to symmetric (close to normal). This answers guiding question 1.

**Step 4: Price by quality and by neighborhood.** Boxplots of `SalePrice` grouped by `OverallQual`, and by `Neighborhood` sorted by median price.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="OverallQual", y="SalePrice", color=BLUE)
plt.title("SalePrice by overall quality")
plt.xlabel("OverallQual (1 = very poor, 10 = very excellent)")
plt.ylabel("Sale price ($)")
plt.gca().yaxis.set_major_formatter(price_fmt)
plt.show()
display(df.groupby("OverallQual")["SalePrice"].agg(["count", "median", "std"]))

order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x="Neighborhood", y="SalePrice", order=order, color=BLUE)
plt.xticks(rotation=90)
plt.title("SalePrice by neighborhood (sorted by median)")
plt.xlabel("Neighborhood")
plt.ylabel("Sale price ($)")
plt.gca().yaxis.set_major_formatter(price_fmt)
plt.show()

nb = df.groupby("Neighborhood")["SalePrice"].median().sort_values()
print("Lowest 3 medians:\n", nb.head(3), "\nHighest 3 medians:\n", nb.tail(3))

### What I learned
- **Quality:** the median price rises at every step: about $50k at quality 1, $86k at 3, $160k at 6, $270k at 8 and $432k at 10. This answers guiding question 3.
- The **spread grows with quality**: the standard deviation is about $15k at quality 1 but about $160k at quality 10, so top-quality houses vary a lot in price.
- Caution: quality 1 and 2 have only 2 and 3 houses, so those boxes are not reliable.
- **Neighborhood:** the highest medians are NridgHt ($315,000), NoRidge ($301,500) and StoneBr ($278,000). The lowest are MeadowV ($88,000), IDOTRR ($103,000) and BrDale ($106,000). The top median is about 3.5 times the lowest, so location clearly matters (guiding question 4).

**Step 4: Counts.** Count plots of `OverallQual` and `HouseStyle` to see how the houses are spread across categories.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.countplot(data=df, x="OverallQual", color=BLUE, ax=axes[0])
axes[0].set(title="Number of houses by OverallQual", xlabel="OverallQual", ylabel="Count")

sns.countplot(data=df, x="HouseStyle", order=df["HouseStyle"].value_counts().index,
              color=BLUE, ax=axes[1])
axes[1].set(title="Number of houses by HouseStyle", xlabel="HouseStyle", ylabel="Count")

plt.tight_layout()
plt.show()

### What I learned
- About 75% of houses (1,090 of 1,460) have quality 5, 6 or 7. Very low and very high quality houses are rare, which is why their boxplots are less stable.
- `1Story` (726) and `2Story` (445) make up about 80% of all houses. The other styles have few examples.

## 6. Bivariate and multivariate analysis

**Step 5: Correlation and covariance.** Find the numerical features most related to `SalePrice`, compare covariance with correlation for the top 3, check Spearman for ordinal columns, and look at features that are correlated with each other.

In [ ]:
corr_cols = [c for c in df.select_dtypes(include="number").columns if c != "Id"]
cov_matrix = df[corr_cols].cov()
corr_matrix = df[corr_cols].corr()

top_corr = corr_matrix["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False)
top10 = top_corr.head(10).index.tolist()
display(pd.DataFrame({"abs_corr": top_corr.head(10),
                      "corr": corr_matrix.loc[top10, "SalePrice"]}))

print("Covariance vs correlation for the top 3 features:")
for feat in top10[:3]:
    print(f"{feat:12s} covariance = {cov_matrix.loc[feat, 'SalePrice']:>14,.0f} | "
          f"correlation = {corr_matrix.loc[feat, 'SalePrice']:.3f}")

# Optional: Spearman for ordinal columns (quality labels turned into codes 0..4)
ordinal_df = pd.DataFrame({"OverallQual": df["OverallQual"],
                           "ExterQual": df["ExterQual"].cat.codes,
                           "KitchenQual": df["KitchenQual"].cat.codes,
                           "SalePrice": df["SalePrice"]})
display(pd.DataFrame({"Pearson": ordinal_df.corr(method="pearson")["SalePrice"],
                      "Spearman": ordinal_df.corr(method="spearman")["SalePrice"]}).drop("SalePrice"))

print("\nFeatures correlated with each other:")
for a, b in [("GarageArea", "GarageCars"), ("TotalBsmtSF", "1stFlrSF"), ("GrLivArea", "TotRmsAbvGrd")]:
    print(f"corr({a}, {b}) = {df[a].corr(df[b]):.3f}")

### What I learned
- Top correlations with `SalePrice`: **OverallQual 0.79**, **GrLivArea 0.71**, GarageCars 0.64, GarageArea 0.62, TotalBsmtSF 0.61 (guiding question 2).
- **Why correlation is easier to compare than covariance:** covariance depends on the units. For example, GrLivArea's covariance with price is about 29,581,867 (sq ft × dollars), while OverallQual's is about 86,904 (quality points × dollars). These numbers cannot be compared. Correlation removes the units and always lies between -1 and +1.
- Spearman is slightly higher than Pearson for `OverallQual` (0.81 vs 0.79), so the relationship is steadily increasing even when it is not perfectly straight.
- **Correlated features:** `GarageArea` and `GarageCars` have r = 0.88, and similar pairs are TotalBsmtSF/1stFlrSF (0.82) and GrLivArea/TotRmsAbvGrd (0.83). They measure nearly the same thing (size), so they are not independent drivers of price. This matters because the "top 10" list partly repeats the same information.
- **Correlation is not causation.** Larger houses tend to be higher quality too, so size and quality are tangled together.

**Step 6: Scatter plots.** Plot `GrLivArea`, `TotalBsmtSF` and `YearBuilt` against `SalePrice`. Highlight houses with `GrLivArea > 4000` and a low price.

In [ ]:
flag = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(df["GrLivArea"], df["SalePrice"], s=15, alpha=0.5, color=BLUE, label="Other houses")
axes[0].scatter(df.loc[flag, "GrLivArea"], df.loc[flag, "SalePrice"], s=70, color="red",
                label="GrLivArea > 4000, low price")
for _, r in df[flag].iterrows():
    axes[0].annotate(f"Id {int(r['Id'])}", (r["GrLivArea"], r["SalePrice"]),
                     xytext=(-55, 12), textcoords="offset points", color="red")
axes[0].set(title="GrLivArea vs SalePrice", xlabel="Living area (sq ft)", ylabel="Sale price ($)")
axes[0].legend()

axes[1].scatter(df["TotalBsmtSF"], df["SalePrice"], s=15, alpha=0.5, color=BLUE)
axes[1].set(title="TotalBsmtSF vs SalePrice", xlabel="Basement area (sq ft)", ylabel="Sale price ($)")

axes[2].scatter(df["YearBuilt"], df["SalePrice"], s=15, alpha=0.5, color=BLUE)
axes[2].set(title="YearBuilt vs SalePrice", xlabel="Year built", ylabel="Sale price ($)")

for ax in axes:
    ax.yaxis.set_major_formatter(price_fmt)
plt.tight_layout()
plt.show()

### What I learned
- Price rises with living area and basement area in a roughly straight line, but the spread widens for larger houses.
- Two houses (Id 524 and Id 1299) have more than 4,000 sq ft but sold for only $184,750 and $160,000. They sit far below the trend, so they are suspicious.
- `YearBuilt` has a weaker, looser link to price (r = 0.52): newer houses cost more on average, but there is a lot of overlap.

**Step 6: Pair plots.** Pair plot of `SalePrice`, `GrLivArea`, `TotalBsmtSF`, `GarageArea` and `YearBuilt` with KDE on the diagonal. Then the same plot coloured by `CentralAir`.

In [ ]:
selected = ["SalePrice", "GrLivArea", "TotalBsmtSF", "GarageArea", "YearBuilt"]

g = sns.pairplot(df[selected], diag_kind="kde", plot_kws={"alpha": 0.4, "s": 15, "color": BLUE})
g.fig.suptitle("Pair plot of key features", y=1.02)
plt.show()

g2 = sns.pairplot(df[selected + ["CentralAir"]], hue="CentralAir", diag_kind="kde",
                  plot_kws={"alpha": 0.4, "s": 15})
g2.fig.suptitle("Pair plot coloured by CentralAir", y=1.02)
plt.show()

### What I learned
- Every size feature shows a positive relationship with `SalePrice`, and the KDE on the diagonal confirms that price, living area and basement area are right skewed.
- In the second plot, look at whether the houses without central air (`N`) sit at older years and lower prices. Write down what you see here.

## Step 7: Correlation heatmap
One picture of how the 10 strongest features relate to `SalePrice` and to each other (upper triangle hidden to avoid repeating values).

In [ ]:
corr_top = df[["SalePrice"] + top10].corr()
mask = np.triu(np.ones_like(corr_top, dtype=bool))

plt.figure(figsize=(11, 8))
sns.heatmap(corr_top, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Top 10 features correlated with SalePrice")
plt.tight_layout()
plt.show()

### What I learned
- The `SalePrice` row is the one to read first: `OverallQual` is the darkest, then `GrLivArea`.
- Dark blocks between features (garage area and garage cars, basement area and first-floor area, living area and room count) show groups of features that carry similar information.
- All 10 features are positively correlated with price. None of the top 10 is a negative correlation.

## 7. Outlier analysis

**Step 8: Detect outliers with two methods.** Compare the IQR rule (1.5 × IQR beyond the quartiles) with the z-score rule (|z| > 3) for `GrLivArea` and `SalePrice`.

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)]

def zscore_outliers(series, threshold=3):
    z = (series - series.mean()) / series.std()
    return series[z.abs() > threshold]

flags, rows = {}, []
for col in ["GrLivArea", "SalePrice"]:
    a, b = iqr_outliers(df[col]), zscore_outliers(df[col])
    flags[col] = (a.index, b.index)
    rows.append({"variable": col, "IQR flagged": len(a), "z-score flagged": len(b),
                 "overlap": len(a.index.intersection(b.index))})
display(pd.DataFrame(rows).set_index("variable"))

### What I learned
- `SalePrice`: IQR flags **61** houses, z-score flags **22**, and all 22 are also in the IQR list.
- `GrLivArea`: IQR flags **31**, z-score flags **16**, and all 16 overlap.
- **Why the z-score flags fewer points on a skewed variable:** the z-score uses the mean and standard deviation, and both are inflated by the same long right tail they are trying to detect. For `SalePrice`, mean + 3 standard deviations is about $419,000. The IQR rule uses quartiles, which the tail does not move, so its upper fence is only about $340,000. On right-skewed data many legitimately expensive houses fall beyond that fence.
- So "flagged" does not mean "wrong". Both methods only tell us where to look.

**Step 8: See the flagged points.** Boxplots with outliers in red, and a scatter plot showing which points each method flagged.

In [ ]:
iqr_idx = flags["GrLivArea"][0].union(flags["SalePrice"][0])
z_idx = flags["GrLivArea"][1].union(flags["SalePrice"][1])
red_dot = {"marker": "o", "markerfacecolor": "red", "markeredgecolor": "red", "markersize": 4}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes[:2], ["GrLivArea", "SalePrice"]):
    sns.boxplot(y=df[col], color=BLUE, flierprops=red_dot, ax=ax)
    ax.set_title(f"{col}: outliers in red")

axes[2].scatter(df["GrLivArea"], df["SalePrice"], s=12, alpha=0.4, color="grey", label="Not flagged")
axes[2].scatter(df.loc[iqr_idx, "GrLivArea"], df.loc[iqr_idx, "SalePrice"],
                s=25, color=ORANGE, label="Flagged by IQR")
axes[2].scatter(df.loc[z_idx, "GrLivArea"], df.loc[z_idx, "SalePrice"],
                s=25, color="red", label="Flagged by IQR and z-score")
axes[2].set(title="GrLivArea vs SalePrice: flagged points",
            xlabel="Living area (sq ft)", ylabel="Sale price ($)")
axes[2].yaxis.set_major_formatter(price_fmt)
axes[2].legend()
plt.tight_layout()
plt.show()

### What I learned
- Most flagged points are big or expensive houses that still follow the upward trend, so they are real sales and not errors.
- The points that do not follow the trend are the very large houses with low prices. These are the ones to look at closely.

**Step 8: Decide.** Look at every house over 4,000 sq ft. Remove the ones that look like non-typical sales and keep the rest, and compare the correlation before and after.

In [ ]:
suspicious = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)

big = df.loc[df["GrLivArea"] > 4000,
             ["Id", "GrLivArea", "SalePrice", "OverallQual", "SaleType", "SaleCondition"]].copy()
big["removed"] = suspicious[big.index]
display(big)

df_clean = df.loc[~suspicious].copy()
print("df shape:", df.shape, "-> df_clean shape:", df_clean.shape)
print(f"corr(GrLivArea, SalePrice) before: {df['GrLivArea'].corr(df['SalePrice']):.3f} | "
      f"after: {df_clean['GrLivArea'].corr(df_clean['SalePrice']):.3f}")

### What I learned
Four houses are larger than 4,000 sq ft, all with quality 10:

| Id | GrLivArea | SalePrice | SaleCondition | Decision |
|---|---|---|---|---|
| 692 | 4,316 | $755,000 | Normal | **Keep**: price matches size and quality |
| 1183 | 4,476 | $745,000 | Abnorml | **Keep**: price matches size and quality |
| 524 | 4,676 | $184,750 | Partial | **Remove**: huge but cheap |
| 1299 | 5,642 | $160,000 | Partial | **Remove**: huge but cheap |

- The two cheap ones are both `Partial` sales of new homes (a home not finished when assessed), so the price likely does not reflect the finished house. They are unusual sales, not typical market behaviour (guiding question 5).
- `df_clean` has **1,458 rows**. Removing just these two raises the correlation of `GrLivArea` with `SalePrice` from **0.709 to 0.735**, which shows how much two points distort a trend line.
- This is a judgement call based on two houses. The removal is documented here so it can be questioned.

## 8. Dashboard

**Step 10: Final dashboard.** A 2 × 2 figure on `df_clean`: average price by year built, top 10 neighborhoods, price distribution with mean and median, and living area vs price coloured by quality. It is saved as `eda_dashboard.png`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: line plot
avg_by_year = df_clean.groupby("YearBuilt")["SalePrice"].mean()
axes[0, 0].plot(avg_by_year.index, avg_by_year.values, color=BLUE, label="Average price")
axes[0, 0].set(title="Average price by year built", xlabel="Year built", ylabel="Average sale price ($)")
axes[0, 0].yaxis.set_major_formatter(price_fmt)
axes[0, 0].legend()

# Panel 2: horizontal bars, top 10 neighborhoods
top_nb = df_clean.groupby("Neighborhood")["SalePrice"].mean().sort_values().tail(10)
axes[0, 1].barh(top_nb.index, top_nb.values, color=BLUE)
axes[0, 1].set(title="Top 10 neighborhoods by average price",
               xlabel="Average sale price ($)", ylabel="Neighborhood")
axes[0, 1].xaxis.set_major_formatter(price_fmt)

# Panel 3: histogram + KDE with mean and median
sns.histplot(df_clean["SalePrice"], kde=True, bins=40, color=BLUE, ax=axes[1, 0])
axes[1, 0].axvline(df_clean["SalePrice"].mean(), color="red", linestyle="--",
                   label=f"Mean: ${df_clean['SalePrice'].mean():,.0f}")
axes[1, 0].axvline(df_clean["SalePrice"].median(), color="green", linestyle="--",
                   label=f"Median: ${df_clean['SalePrice'].median():,.0f}")
axes[1, 0].set(title="SalePrice distribution", xlabel="Sale price ($)", ylabel="Number of houses")
axes[1, 0].xaxis.set_major_formatter(price_fmt)
axes[1, 0].legend()

# Panel 4: scatter coloured by quality
sns.scatterplot(data=df_clean, x="GrLivArea", y="SalePrice", hue="OverallQual",
                palette="viridis", s=25, alpha=0.8, ax=axes[1, 1])
axes[1, 1].set(title="Living area vs price, coloured by quality",
               xlabel="Living area (sq ft)", ylabel="Sale price ($)")
axes[1, 1].yaxis.set_major_formatter(price_fmt)
axes[1, 1].legend(title="OverallQual", ncol=2, fontsize=8)

fig.suptitle("House Price EDA Summary", fontsize=18, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("eda_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

# Download the picture to your computer (works in Colab)
try:
    from google.colab import files
    files.download("eda_dashboard.png")
except ImportError:
    pass

### What I learned
- Newer houses cost more on average, but the line is jagged for old years because each year has few houses.
- NoRidge, NridgHt and StoneBr lead the average-price ranking among neighborhoods.
- In the histogram the mean line sits to the right of the median line, which is the right skew again.
- In the scatter plot, the colours shift from dark (low quality) to bright (high quality) as price rises, so size and quality both matter and move together.

## 9. Conclusions

### Expected insights, checked against the data
| # | Expected insight | Result |
|---|---|---|
| 1 | `SalePrice` is right skewed; log makes it close to normal | **Confirmed**: skewness 1.88 → 0.12 |
| 2 | `OverallQual` and `GrLivArea` are among the strongest correlates | **Confirmed**: r = 0.79 and 0.71 |
| 3 | Price rises steadily with quality, with more spread at the top | **Confirmed**: median $50k → $432k, standard deviation $15k → $160k (small groups at quality 1-2) |
| 4 | A few neighborhoods have clearly higher medians | **Confirmed**: NridgHt $315k, NoRidge $301k, StoneBr $278k vs MeadowV $88k |
| 5 | Very large, cheap houses are unusual sales that distort trend lines | **Confirmed**: Ids 524 and 1299 (`Partial` sales); removing them lifts r from 0.709 to 0.735 |
| 6 | Many "missing" values are really "feature absent" | **Confirmed**: pool, alley, fence, fireplace, garage and basement blanks match zero areas or counts; only `LotFrontage` (259) and `Electrical` (1) are truly missing |

### Limitations
- Only 1,460 houses from one city (Ames, Iowa), so the results may not apply elsewhere.
- Correlation is not causation. Size, quality and garage features overlap, so their effects cannot be separated by EDA alone.
- Some groups are small (quality 1-2, neighborhoods like Blueste with 2 houses), so their summaries are unreliable.
- Cleaning involved choices (filling with "None", median by neighborhood, removing two houses). A different choice could change some numbers slightly.
- No machine learning was used, as required. A natural next step would be a regression model on `log(SalePrice)`.